# OpenAI SDK Agentic AI: Handoff and `as_tool` Usage

This document demonstrates how to use **Agentic AI** with the OpenAI-compatible SDK, focusing on:
- Task handoff between agents
- Using agents as tools (`as_tool`)

This notebook provides API examples using **.....** and **DeepSeek**, both compatible with OpenAI-style APIs.

This is an educational and open-source example generated with assistance from DeepSeek and validated through testing.

For original reference (in Chinese):
.........................................................


## Concept: Handoff

In multi-agent systems, **handoff** allows one agent to transfer both:
- Task responsibility
- Workflow state

### Key Rules:
- An agent can only hand off to **one agent at a time**.
- Tools are **not transferred** — each agent keeps its own tools.
- Think of it like one employee passing a task to another.

Example:
- Agent A → can hand off to Agent B or C
- Once transferred, the receiving agent continues execution


In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent, Runner,
    OpenAIChatCompletionsModel, set_tracing_disabled
)

# --- Setup ---
load_dotenv(override=True)
set_tracing_disabled(True)

client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

model = OpenAIChatCompletionsModel(
    model="deepseek-chat",
    openai_client=client
)

# Specialist agents
billing_agent = Agent(
    name="billing_agent",
    instructions="You are a billing expert.",
    model=model
)

refund_agent = Agent(
    name="refund_agent",
    instructions="You are a refund expert.",
    model=model
)

# Triage agent
triage_agent = Agent(
    name="triage_agent",
    instructions="Route user requests appropriately.",
    handoffs=[billing_agent, refund_agent],
    model=model
)

async def main():
    result = await Runner.run(
        triage_agent,
        "I was charged twice for my subscription."
    )
    print(result.final_output)

await main()


## Concept: Agent as Tool (`as_tool`)

Agents can also be converted into tools and called by other agents.

### Key Differences:
- Handoff = transfer responsibility
- `as_tool` = function-style usage

### Important:
- One agent can use **multiple tools in parallel**
- Parallel execution improves performance


In [ ]:
import asyncio
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent, Runner, SQLiteSession,
    OpenAIChatCompletionsModel, set_tracing_disabled,
    ModelSettings
)

load_dotenv(override=True)
set_tracing_disabled(True)

client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

model = OpenAIChatCompletionsModel(
    model="deepseek-chat",
    openai_client=client
)

# Define agents
finance_agent = Agent(
    name="Finance_Analyst",
    instructions="Analyze financial health.",
    model=model
)

market_agent = Agent(
    name="Market_Analyst",
    instructions="Analyze market position.",
    model=model
)

# Convert to tools
finance_tool = finance_agent.as_tool(
    tool_name="get_financial_analysis",
    tool_description="Financial analysis"
)

market_tool = market_agent.as_tool(
    tool_name="get_market_analysis",
    tool_description="Market analysis"
)

# Manager agent
manager_agent = Agent(
    name="Research_Manager",
    instructions="Call both tools in parallel.",
    tools=[finance_tool, market_tool],
    model=model,
    model_settings=ModelSettings(parallel_tool_calls=True)
)

session = SQLiteSession("demo_user", "conversations.db")

async def main():
    result = await Runner.run(
        manager_agent,
        "Analyze Tesla",
        session=session
    )
    print(result.final_output)

await main()


## Notes

- Agentic AI SDKs are still evolving
- Concepts are more important than code
- You can implement your own:
  - session management
  - handoff logic
  - tool orchestration

Using the SDK simplifies development for smaller systems


## Contact

- Business / HR: yucongcai_business@outlook.com
- Research: yucongcai_research@outlook.com
